In [ ]:
import pandas as pd
import numpy as np


In [21]:
import requests
import pickle
import io

In [22]:
url = "https://huggingface.co/datasets/zyllab/TTMs_on_MG/resolve/main/rolling_prediction.pkl"
response = requests.get(url)
data = pickle.load(io.BytesIO(response.content))
df = pd.DataFrame(data)

In [23]:
df_filtered = df[
    (df['tau'] == 60) &
    (df['frac'] == 30) &
    (df['location'] == "uniform")
]

In [37]:
np.array(df_filtered['Pred'].tolist())[0,0]

array([[0.81824262, 0.81219578, 0.80876928, ..., 0.94621923, 0.94440416,
        0.94261399],
       [0.78630662, 0.78131608, 0.77889156, ..., 1.01708266, 1.012917  ,
        1.00772385],
       [0.75753008, 0.75348487, 0.75198744, ..., 0.96446347, 0.96077733,
        0.95742716],
       ...,
       [0.50834752, 0.48994448, 0.47671844, ..., 0.98775547, 0.98813191,
        0.9881116 ],
       [0.49102267, 0.47505774, 0.46482735, ..., 0.96021184, 0.96051356,
        0.96014735],
       [0.48353659, 0.47133738, 0.46494257, ..., 0.96367158, 0.96312522,
        0.96321374]], shape=(906, 1632))

In [39]:
def MG_generate_interp(gamma=0.1,beta=0.2,tau=23,theta=1,n=10,x0=0.2,N=1000000,delta=0.01,past_val=0.1):
  def MG_eq (x,x_pre):
    return x_pre * beta * (theta**n)/(theta**n+x_pre**n)-gamma*x

  def MG_rk4(x,x_pre,x_pre_f):
    interplot = (x_pre+x_pre_f)/2
    k1 = MG_eq(x,x_pre)
    k2 = MG_eq(x+delta*k1/2,interplot)
    k3 = MG_eq(x+delta*k2/2,interplot)
    k4 = MG_eq(x+delta*k3,x_pre_f)
    return x+delta*(k1+2*k2+2*k3+k4)/6

  past_len = int(np.floor(tau/delta))
  x_past = np.zeros(past_len+N+1)+past_val
  x = x0
  X = np.zeros(N+1)
  T = np.zeros(N+1)
  time = 0

  for i in range(N+1):
    X[i] = x
    x_pre = x_past[i]
    x_pre_f = x_past[i+1]
    x_delta = MG_rk4(x=x,x_pre=x_pre,x_pre_f = x_pre_f)
    x_past[i+past_len] = x_delta
    T[i] = time
    time += delta
    x = x_delta

  return T,X

In [41]:
from sklearn.metrics import mutual_info_score
import nolds

In [42]:
def mf_value(x):
    n = len(x)
    fft_vals = np.fft.rfft(x,n)
    psd = np.abs(fft_vals)**2
    freqs = np.fft.rfftfreq(n)
    # Skip DC component at index 0
    mean_freq = np.sum(freqs[1:] * psd[1:]) / np.sum(psd[1:])
    return mean_freq

def compute_ami_lag(x, max_lag=100, bins=32):
    """
    Estimate optimal embedding lag via average mutual information (AMI).
    Returns list of AMI values and the first local minimum lag.
    """
    # Discretize data into bins
    hist, bin_edges = np.histogram(x, bins=bins)
    digitized = np.digitize(x, bin_edges[:-1])
    
    ami = []
    for lag in range(1, max_lag):
        x1 = digitized[:-lag]
        x2 = digitized[lag:]
        mi = mutual_info_score(x1, x2)
        ami.append(mi)

    # Find the first local minimum
    for i in range(1, len(ami) - 1):
        if ami[i] < ami[i-1] and ami[i] < ami[i+1]:
            return i + 1, ami  # +1 because lag starts at 1

    return np.argmin(ami) + 1, ami  # fallback if no local min
def compute_le(x):
    mf = mf_value(x)
    min_tsep = int(round(1/mf))
    lag,_ = compute_ami_lag(x,bins = 100)
    #print(min_tsep,lag)
    le = nolds.lyap_r(x,min_tsep=min_tsep,lag=lag)
    return le

In [59]:
np.random.seed(0)
df = pd.DataFrame(columns=['tau','dif_idx','dif_x0'])
for tau in [60,120,200]:
    les_org_diff_idx = []
    _,X = MG_generate_interp(x0 =1,tau=tau)
    for i in range(1,50):
        start = i*40+6000
        X_tru= X[::100][start:start+500]
        le_org_diff_idx = compute_le(X_tru)
        les_org_diff_idx.append(le_org_diff_idx)

    les_org_diff_x0 = []

    for i in range(1,50):
        print(i)
        _,X = MG_generate_interp(x0 =(i+1)/5,tau=tau)
        X_tru= X[::100][7800:8300]
        le_org_diff_x0 = compute_le(X_tru)
        les_org_diff_x0.append(le_org_diff_x0)
    df.loc[len(df)] = [tau,[les_org_diff_idx],[les_org_diff_x0]]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


In [60]:
df.to_pickle("les_org.pkl")

In [63]:
les_df = pd.read_pickle("les_org.pkl")

In [64]:
tau = 60
les_df_filtered = les_df[
    (les_df['tau'] == tau)
]

In [67]:
np.array(les_df_filtered['dif_idx'].tolist())[0,0]

array([0.00633995, 0.00879456, 0.01045657, 0.03021374, 0.04514954,
       0.03825524, 0.00428101, 0.00444462, 0.001916  , 0.00321165,
       0.00733749, 0.0066502 , 0.00391314, 0.00408711, 0.00252403,
       0.00240437, 0.00316348, 0.0061445 , 0.0099344 , 0.00988958,
       0.01050368, 0.01036249, 0.00924928, 0.00837401, 0.00853472,
       0.00851717, 0.00662378, 0.01068796, 0.00870582, 0.00743319,
       0.00763737, 0.03237838, 0.02630164, 0.00393364, 0.0040471 ,
       0.00786596, 0.00521165, 0.00554631, 0.00562696, 0.00565529,
       0.0061685 , 0.00719581, 0.0069064 , 0.00689992, 0.00645289,
       0.00554373, 0.03346242, 0.0311766 , 0.00717504])